# Support Vector Machine (SVM)
Owner: Anupama Singh

Predicts the 0/1 label with a support vector machine. With the RBF kernel it can draw curved (non-linear) boundaries between high- and low-growth students. Extension beyond the paper.

## 1. Setup and data
Same data as the paper and as the rest of the team: 706 labeled students with no missing values, the paper's 16 features, two targets. The folds are identical to Megha's Random Forest and XGBoost notebooks.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, mean_absolute_error
from sklearn.svm import SVC
# Data: local copy in data/ if present, otherwise download from the ECRI repository
LOCAL = Path("../data")
URL = "https://raw.githubusercontent.com/ecri-data/ECRI-Dataset/main/ECRI"
src = LOCAL if (LOCAL / "ECRIDataSet.csv").exists() else URL

# Outputs: the repo's results/ folder when run locally, ./results in Colab
RESULTS = Path("../results") if Path("../results").exists() else Path("results")
RESULTS.mkdir(exist_ok=True)

# The paper's 16 features (Table 3) and the two Woodcock targets
FEATURES = ["Gender", "Tx", "Age1b", "Tier2_N", "grp_rate", "rcmistot", "gnrl_fid",
            "TKPctCrct", "NWFcls", "NWFwrc", "ORFwc", "SAwrS", "SAsrS", "SAtoS",
            "RMwidRS", "RMwdaRS"]
TARGETS = {"Word Identification": "RMwidRS", "Word Attack": "RMwdaRS"}

df = pd.read_csv(f"{src}/ECRIDataSet.csv")
raw = pd.read_csv(f"{src}/ECRIRawData.csv", low_memory=False)

# Spring scores are used ONLY to compute the gain (regression target), never as features
spring = raw[raw["Time"] == 2.0].set_index("StuID")[list(TARGETS.values())].add_suffix("_spring")

# Labeled students with no missing feature values, as in the paper (706 students)
data = (df[df["RMwidRS_target"].notna()]
        .dropna(subset=FEATURES)
        .join(spring, on="StuID")
        .reset_index(drop=True))
for score in TARGETS.values():
    data[f"{score}_gain"] = data[f"{score}_spring"] - data[score]

X = data[FEATURES]
print(f"Students: {len(data)} | Schools: {data['SchlID'].nunique()} | Features: {len(FEATURES)}")

Students: 706 | Schools: 42 | Features: 16


## 2. The model
- Features are standardized inside each training fold (SVMs need this).
- **Kernel:** `linear` draws a straight boundary; `rbf` can draw curved ones.
- **C:** a larger `C` fits the training data more tightly (risk of overfitting); a smaller `C` gives a smoother boundary.
- **gamma** (RBF only): how curved the boundary can be; small gamma = smooth, large gamma = wiggly.
- **24 settings:** linear kernel with 6 values of C (0.01 to 3), and RBF kernel with 6 values of C (0.1 to 30) × 3 values of gamma.
  This medium grid was chosen after comparing grids of 7, 24, 48, 102 and 150 settings in `anupama_svm_tuning_check.ipynb`: it gave the best
  ROC AUC; bigger grids started to overfit the tuning.
- Kernel, `C` and gamma are **tuned together inside each training fold** with an inner 4-fold cross-validation (best ROC AUC). The test fold is never used for this choice.
- `probability=True` makes the SVM output probabilities, needed for ROC and precision-recall curves.

In [2]:
import warnings
# Newer scikit-learn versions (1.8+) announce renamed settings; the code below works on all versions
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")
warnings.filterwarnings("ignore", message="Inconsistent values", category=UserWarning)

MODEL_NAME = "SVM"
KIND = "classifier"
# 24 settings: linear kernel with 6 values of C, RBF kernel with 6 values of C x 3 values of gamma
SETTINGS = {f"linear, C={c}": ("linear", c, "scale") for c in [0.01, 0.03, 0.1, 0.3, 1, 3]}
SETTINGS.update({f"rbf, C={c}, gamma={g}": ("rbf", c, g)
                 for c in [0.1, 0.3, 1, 3, 10, 30] for g in ["scale", 0.01, 0.03]})


def build_model(name):
    kernel, c, gamma = SETTINGS[name]
    return make_pipeline(StandardScaler(), SVC(kernel=kernel, C=c, gamma=gamma, probability=True, random_state=0))


def pick_setting(X_train, y_train, groups_train):
    # Inner 4-fold grouped CV on the training folds only (best ROC AUC); the test fold is never used
    scores = {name: cross_val_score(build_model(name), X_train, y_train, cv=GroupKFold(4),
                                    groups=groups_train, scoring="roc_auc").mean()
              for name in SETTINGS}
    return max(scores, key=scores.get)

## 3. 5-fold cross-validation
- **School split (main):** `GroupKFold(5)` on `SchlID`. Every school sits in exactly one test fold, so each student is scored by a model that never saw their school.
- **Student split:** `GroupKFold(5)` on `StuID`, so a school can appear in both training and test.

Each round trains on 4 folds (about 565 students) and predicts the 5th (about 141). The prediction for every student is saved so the evaluation notebook can compute all metrics and curves.

In [3]:
SPLITS = {"school": "SchlID", "student": "StuID"}


def cross_validate(score, split):
    """5-fold grouped cross-validation. Returns one row per student with the out-of-fold prediction."""
    groups = data[SPLITS[split]]
    y = data[f"{score}_target"] if KIND == "classifier" else data[f"{score}_gain"]
    rows = []
    for fold, (train_idx, test_idx) in enumerate(GroupKFold(n_splits=5).split(X, y, groups), start=1):
        setting = pick_setting(X.iloc[train_idx], y.iloc[train_idx], groups.iloc[train_idx])
        model = build_model(setting).fit(X.iloc[train_idx], y.iloc[train_idx])
        test = data.iloc[test_idx]
        prediction = model.predict_proba(test[FEATURES])[:, 1]    # probability of label 1
        pred_label = (prediction >= 0.5).astype(int)
        rows.append(pd.DataFrame({
            "split": split, "target": score, "model": MODEL_NAME, "fold": fold,
            "StuID": test["StuID"].values, "SchlID": test["SchlID"].values, "setting": setting,
            "true_label": test[f"{score}_target"].astype(int).values,
            "true_gain": test[f"{score}_gain"].values,
            "score": prediction,          # probability of label 1
            "pred_label": pred_label,
        }))
    return pd.concat(rows, ignore_index=True)


predictions = pd.concat([cross_validate(score, split)
                         for split in SPLITS for score in TARGETS.values()], ignore_index=True)
print("Predictions:", len(predictions), "rows (706 students x 2 targets x 2 splits)")

Predictions: 2824 rows (706 students x 2 targets x 2 splits)


## 4. Quick summary
Mean over the 5 folds. The full set of metrics and curves (precision, recall, ROC and PR curves, squared error) is in `anupama_baseline_evaluation.ipynb`.

In [4]:
rows = []
for (split, score, fold), g in predictions.groupby(["split", "target", "fold"]):
    row = {"split": split, "target": score, "fold": fold,
           "ROC AUC": roc_auc_score(g["true_label"], g["score"]),
           "PR-AUC": average_precision_score(g["true_label"], g["score"]),
           "Accuracy": accuracy_score(g["true_label"], g["pred_label"])}
    rows.append(row)
fold_results = pd.DataFrame(rows)

summary = (fold_results.drop(columns="fold").groupby(["split", "target"])
           .agg(["mean", "std"]).round(3)
           .rename(index={v: k for k, v in TARGETS.items()}, level=1))
summary

ROC AUC        PR-AUC        Accuracy       
                               mean    std   mean    std     mean    std
split   target                                                          
school  Word Attack           0.593  0.058  0.658  0.076    0.585  0.063
        Word Identification   0.684  0.053  0.697  0.039    0.637  0.050
student Word Attack           0.626  0.035  0.697  0.063    0.603  0.024
        Word Identification   0.699  0.070  0.721  0.097    0.650  0.075

## Setting chosen in each fold
Picked by the inner cross-validation inside each training fold.

In [5]:
(predictions.groupby(["split", "target", "fold"])["setting"].first()
 .unstack("fold").rename(index={v: k for k, v in TARGETS.items()}, level=1))

fold                                             1                      2  \
split   target                                                              
school  Word Attack          rbf, C=3, gamma=scale            linear, C=3   
        Word Identification   rbf, C=3, gamma=0.01   rbf, C=3, gamma=0.01   
student Word Attack           rbf, C=3, gamma=0.03   rbf, C=3, gamma=0.01   
        Word Identification   rbf, C=3, gamma=0.03  rbf, C=10, gamma=0.01   

fold                                             3                       4  \
split   target                                                               
school  Word Attack          rbf, C=10, gamma=0.01  rbf, C=0.1, gamma=0.01   
        Word Identification   rbf, C=3, gamma=0.01    rbf, C=1, gamma=0.01   
student Word Attack           rbf, C=1, gamma=0.01    rbf, C=3, gamma=0.01   
        Word Identification  rbf, C=10, gamma=0.01    rbf, C=3, gamma=0.03   

fold                                             5  
split   target                                      
school  Word Attack          rbf, C=30, gamma=0.01  
        Word Identification  rbf, C=10, gamma=0.01  
student Word Attack                  linear, C=0.3  
        Word Identification  rbf, C=10, gamma=0.01

### How often each kernel won
If the linear kernel wins most folds, a curved boundary does not help on this data.

In [6]:
chosen = predictions.groupby(["split", "target", "fold"])["setting"].first().reset_index()
chosen["kernel"] = chosen["setting"].str.split(",").str[0]
chosen.groupby(["split", "kernel"]).size().unstack(fill_value=0)

kernel,linear,rbf
split,,
school,1,9
student,1,9


## Save the results

In [7]:
predictions.to_csv(RESULTS / "svm_cv_predictions.csv", index=False)
fold_results.to_csv(RESULTS / "svm_cv_fold_results.csv", index=False)
print(f"Saved svm_cv_predictions.csv and svm_cv_fold_results.csv to {RESULTS}/")

Saved svm_cv_predictions.csv and svm_cv_fold_results.csv to ../results/
